In [0]:
#Reading the CUSTOMERS table
customers_df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/de_project_/bronze/raw_historical_data_parent_company/dim_customers.csv")

In [0]:
display(customers_df)

customer_code,customer,market,platform,channel
70002017,Sporty Essentials,India,Brick & Mortar,Retailer
70002018,PlayMore Sports,India,Brick & Mortar,Retailer
90002001,SportsHub,India,E-Commerce,Retailer
90002002,PowerPlay Sports,India,Brick & Mortar,Retailer
90002003,Winning Gear,India,Brick & Mortar,Retailer
90002004,Atlikon Sports India,India,Brick & Mortar,Direct
90002005,Atlikon Official,India,E-Commerce,Direct
90002006,Atlikon Essentials,India,Brick & Mortar,Direct
90002007,SportsMart,India,Brick & Mortar,Retailer
90002008,FitnessWorld,India,E-Commerce,Retailer


In [0]:
#Write it as delta table in BRONZE SCHEMA
customers_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("de_project_.bronze.customers_parent")

READ and WRITE the GROSS_PRICE table

In [0]:
gross_price_df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/de_project_/bronze/raw_historical_data_parent_company/dim_gross_price.csv")

In [0]:
display(gross_price_df)

product_code,price_inr,year
ARCHDDE20D,2750,2024
ARCH158F41,5740,2024
ARCHAFF0E4,4610,2024
ARCH6B94F7,3910,2024
ARCH5D1FE7,1610,2024
ARCH7B49A9,1610,2024
ARCH497D34,1100,2024
ARCHE71D79,5830,2024
ARCHDD8749,3930,2024
BADMC045D4,4500,2024


In [0]:
gross_price_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("de_project_.bronze.gross_price_parent")

READ and WRITE the PRODUCTS table

In [0]:
products_df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/de_project_/bronze/raw_historical_data_parent_company/dim_products.csv")

In [0]:
display(products_df)

product_code,division,category,product,variant
ARCHDDE20D,Archery,Arrows,PX Carbon Arrow Set,12 Pack
ARCH158F41,Archery,Finger Tab,LX Leather Finger Tab,Universal
ARCHAFF0E4,Archery,Bow,AX Precision Recurve Bow,26 lbs
ARCH6B94F7,Archery,Bow,AX Precision Recurve Bow,30 lbs
ARCH5D1FE7,Archery,Bow Stringer,BX Bow Stringing Tool,Standard
ARCH7B49A9,Archery,Arm Guard,NX Archery Arm Guard,Medium
ARCH497D34,Archery,Target,TX Foam Archery Target,80 cm
ARCHE71D79,Archery,Arm Guard,NX Archery Arm Guard,Large
ARCHDD8749,Archery,Target,TX Foam Archery Target,60 cm
BADMC045D4,Badminton,Badminton Racket,BX AeroLite Badminton Racket,Head Heavy


In [0]:
products_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("de_project_.bronze.products_parent")


READ and WRITE the ORDERS table

In [0]:
orders_df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/de_project_/bronze/raw_historical_data_parent_company/fact_orders.csv")

In [0]:
display(orders_df)

date,product_code,customer_code,sold_quantity
2024-01-01,ARCHDDE20D,70002017,161
2024-01-01,ARCH158F41,70002017,133
2024-01-01,ARCHAFF0E4,70002017,76
2024-01-01,ARCH6B94F7,70002017,92
2024-01-01,ARCH5D1FE7,70002017,117
2024-01-01,ARCH7B49A9,70002017,36
2024-01-01,ARCH497D34,70002017,98
2024-01-01,ARCHE71D79,70002017,156
2024-01-01,BADM88C384,70002017,28
2024-01-01,BADMA5EBA3,70002017,33


In [0]:
orders_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("de_project_.bronze.orders_parent")

In [0]:
# Read all 4 bronze tables
customers_df = spark.table("de_project_.bronze.customers_parent")
products_df = spark.table("de_project_.bronze.products_parent")
gross_price_df = spark.table("de_project_.bronze.gross_price_parent")
orders_df = spark.table("de_project_.bronze.orders_parent")

# Print all schemas
print("======= CUSTOMERS SCHEMA =======")
customers_df.printSchema()

print("======= PRODUCTS SCHEMA =======")
products_df.printSchema()

print("======= GROSS PRICE SCHEMA =======")
gross_price_df.printSchema()

print("======= ORDERS SCHEMA =======")
orders_df.printSchema()

======= CUSTOMERS SCHEMA =======
root
 |-- customer_code: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)

======= PRODUCTS SCHEMA =======
root
 |-- product_code: string (nullable = true)
 |-- division: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- variant: string (nullable = true)

======= GROSS PRICE SCHEMA =======
root
 |-- product_code: string (nullable = true)
 |-- price_inr: integer (nullable = true)
 |-- year: integer (nullable = true)

======= ORDERS SCHEMA =======
root
 |-- date: date (nullable = true)
 |-- product_code: string (nullable = true)
 |-- customer_code: integer (nullable = true)
 |-- sold_quantity: integer (nullable = true)



In [0]:
print("Customers count:", customers_df.count())
print("Products count:", products_df.count())
print("Gross Price count:", gross_price_df.count())
print("Orders count:", orders_df.count())

Customers count: 18
Products count: 397
Gross Price count: 794
Orders count: 93055


###LOAD THE TABLES TO SILVER SCHEMA

In [0]:
# Read from bronze tables
customers_df = spark.table("de_project_.bronze.customers_parent")
products_df = spark.table("de_project_.bronze.products_parent")
gross_price_df = spark.table("de_project_.bronze.gross_price_parent")
orders_df = spark.table("de_project_.bronze.orders_parent")

In [0]:
# Change the data type of customer code from integer to string in customers and orders tables as we dont do any operations on that column
customers_silver = customers_df.withColumn("customer_code", customers_df["customer_code"].cast("string"))
orders_silver = orders_df.withColumn("customer_code", orders_df["customer_code"].cast("string"))


In [0]:
# Write to Silver schema
customers_silver.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable("de_project_.silver.customers_parent")

products_df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable("de_project_.silver.products_parent")

gross_price_df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable("de_project_.silver.gross_price_parent")

orders_silver.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable("de_project_.silver.orders_parent")

print("Silver tables written successfully!")

Silver tables written successfully!


In [0]:
# Read all 4 bronze tables
customers_df1 = spark.table("de_project_.silver.customers_parent")
products_df1 = spark.table("de_project_.silver.products_parent")
gross_price_df1 = spark.table("de_project_.silver.gross_price_parent")
orders_df1 = spark.table("de_project_.silver.orders_parent")

# Print all schemas
print("======= CUSTOMERS SCHEMA =======")
customers_df1.printSchema()

print("======= PRODUCTS SCHEMA =======")
products_df1.printSchema()

print("======= GROSS PRICE SCHEMA =======")
gross_price_df1.printSchema()

print("======= ORDERS SCHEMA =======")
orders_df1.printSchema()

======= CUSTOMERS SCHEMA =======
root
 |-- customer_code: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)

======= PRODUCTS SCHEMA =======
root
 |-- product_code: string (nullable = true)
 |-- division: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- variant: string (nullable = true)

======= GROSS PRICE SCHEMA =======
root
 |-- product_code: string (nullable = true)
 |-- price_inr: integer (nullable = true)
 |-- year: integer (nullable = true)

======= ORDERS SCHEMA =======
root
 |-- date: date (nullable = true)
 |-- product_code: string (nullable = true)
 |-- customer_code: string (nullable = true)
 |-- sold_quantity: integer (nullable = true)



In [0]:
print("Customers count:", customers_df1.count())
print("Products count:", products_df1.count())
print("Gross Price count:", gross_price_df1.count())
print("Orders count:", orders_df1.count())

Customers count: 18
Products count: 397
Gross Price count: 794
Orders count: 93055


# CREATE A DIMENSIONAL DATE TABLE

In [0]:
from pyspark.sql import functions as F

# start date and end date
start_date = "2024-01-01"
end_date   = "2025-12-01"

# Generate one row per month
df = (
    spark.sql(f"""
        SELECT explode(
            sequence(
                to_date('{start_date}'),
                to_date('{end_date}'),
                interval 1 month
            )
        ) AS month_start_date
    """)
)

# Add analytics columns
df = (
    df
    .withColumn("date_key", F.date_format("month_start_date", "yyyyMM").cast("int"))
    .withColumn("year", F.year("month_start_date"))
    .withColumn("month_name", F.date_format("month_start_date", "MMMM"))
    .withColumn("month_short_name", F.date_format("month_start_date", "MMM"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.quarter("month_start_date")))
    .withColumn("year_quarter", F.concat(F.col("year"), F.lit("-Q"), F.quarter("month_start_date")))
)

# Verify
display(df)

# Save to silver schema
df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("de_project_.silver.dim_date")

print("dim_date table created successfully!")

month_start_date,date_key,year,month_name,month_short_name,quarter,year_quarter
2024-01-01,202401,2024,January,Jan,Q1,2024-Q1
2024-02-01,202402,2024,February,Feb,Q1,2024-Q1
2024-03-01,202403,2024,March,Mar,Q1,2024-Q1
2024-04-01,202404,2024,April,Apr,Q2,2024-Q2
2024-05-01,202405,2024,May,May,Q2,2024-Q2
2024-06-01,202406,2024,June,Jun,Q2,2024-Q2
2024-07-01,202407,2024,July,Jul,Q3,2024-Q3
2024-08-01,202408,2024,August,Aug,Q3,2024-Q3
2024-09-01,202409,2024,September,Sep,Q3,2024-Q3
2024-10-01,202410,2024,October,Oct,Q4,2024-Q4


dim_date table created successfully!
